### <span style="color:rgb(46,139,87)">Optimization for Data Science</span>

### <span style="color:rgb(46,139,87)">M2 MIAGE & MIAGE ID Apprentissage, 2026-2027</span>


# <span style="color:rgb(46,139,87)">Lab - Gradient-type methods</span>


This Jupyter notebook was kindly provided by [Clément W. Royer](https://ligm.univ-eiffel.fr/~croyer/), who taught this course in previous years; it has been slightly adapted since. The current version and the one with solutions can be found [here](https://bastiencavarretta.github.io/teachings/2026-optimization-data-science/).


# <span style="color:rgb(46,139,87)">Introduction</span>

In this session, we introduce gradient descent, arguable the most classical nonlinear optimization method. The goal of this notebook is to provide practical illustration of the theory that will be detailed in the next session. To this end, we will restrict ourselves to convex optimization problems.

#### <span style="color:rgb(46,139,87)">Preliminary remarks</span>

- This notebook and the subsequent ones used in this course mix Python code and text/LaTeX blocks. They can be run offline on any computer where Python and Jupyter are installed, or online using Google Colab (requires a Google account).

- **Start** by launching VS Code, then `ctrl + o` to open the file named `LabODS-1-GradientMethods.ipynb`.

- All code blocks from this notebook are meant to be run in the order that they are given. In particular, the first block below must be run first in order to import the necessary toolboxes. Shortcuts: `ctrl + enter` to execute the cell. `ctrl + shift` to execute the cell and go to the following. Other essential shortcuts [here](https://towardsdatascience.com/jypyter-notebook-shortcuts-bf0101a98330/).

- All the notebooks from this course rely on Python and the NumPy library. A basic yet very useful tutorial on NumPy is freely available
[here](https://sebastianraschka.com/pdf/books/dlb/appendix_f_numpy-intro.pdf).

- To learn something from these computer sessions, it is highly recommended to disable chatbots such as Gemini if you are using Google Colab or Copilot in VSCode.

In [1]:
# Import useful librairies and functions
###############################################

# Plots
%matplotlib inline
import matplotlib.pyplot as plt

from math import sqrt # Square root

# NumPy - Vector and matrix structures
import numpy as np # NumPy library
from numpy.random import multivariate_normal, randn, uniform, choice # Probability distributions

# SciPy - Efficient numerical calculations
from scipy.linalg import norm # Standard norms
from scipy.linalg import toeplitz # Toeplitz matrices
from scipy.linalg import qr # QR matrix decomposition
from scipy.linalg import svdvals # Singular value decomposition (recall Lecture 2!)
from scipy.optimize import check_grad # Numerical check of derivatives
from scipy.optimize import fmin_l_bfgs_b # An efficient minimization routine in moderate dimensions

ModuleNotFoundError: No module named 'matplotlib'

#### <span style="color:rgb(46,139,87)">Useful NumPy routines (check the documentation for more details)</span>

* *transpose* Transpose of a matrix (i.e. a two-dimensional NumPy array), also works for vectors. If X is the variable, X.T also works.
* *matmul* Matrix-matrix product (dimensions permitting). Warning: A * B returns the componentwise product.
* *dot* Matrix-vector product (dimensions permitting), can also be used as inner/scalar product between two vectors of same length.
* *np.ones((m,n))* m-by-n matrix with all components being equal to 1. 
* *np.zeros((m,n))* m-by-n matrix with all components being equal to 0.
* *np.identity(n)*  n-by-n identity matrix (ones on the diagonal, zeroes everywhere else).
* *np.pi* $\pi$.
* *np.inf* Infinite number representation.
* *np.log* Logarithm operator applied componentwise to a NumPy array.
* *np.exp* Exponential operator applied componentwise to a NumPy array.
* *np.sum* Sums the components of NumPy array (for matrices, sums along one dimension)
* *np.maximum(u,v)* Returns a NumPy array with coordinates $max(u_i,v_i)$, where $u_i$ and $v_i$ are the coordinates of $u$ and $v$, respectively.;
* *np.concatenate* gathers NumPy arrays (vectors or matrices) with compatibles dimensions.
* For any NumPy array *a*, *a.shape* returns the dimension(s) of this array (useful to build another array with the same dimensions).

# <span style="color:rgb(46,139,87)"> Part 1. Gradient descent on a strongly convex quadratic</span>

## <span style="color:rgb(46,139,87)">1.1 - Problem and analysis</span>

To begin this notebook, we will study the following quadratic optimization problem:

$$
    \mathrm{minimize}_{\mathbf{w} \in \mathbb{R}^d} f(\mathbf{w}):=\tfrac{1}{2}\mathbf{w}^T \mathbf{C} \mathbf{w},
$$

where $\mathbf{C} \in \mathbb{R}^d$ is a symmetric, positive semidefinite matrix with eigenvalues
$$
    0 < \lambda_1 \le \dots \le \lambda_d.
$$
This simple setting will allow us to illustrate the performance of gradient-type methods in a convex setting. 

### <span style="color:rgb(46,139,87)">Question 1 - Back to lecture 1</span> 

1) *The function $f$ is $\mathcal{C}^1$ and convex. How do we characterize the solutions of this problem using the gradient of $f$ ?*

2) *The function $f$ is in fact $\lambda_1$-strongly convex. Which additional property does this imply on the optimal solutions?*

3) *Using the formula $\nabla f(\mathbf{w}) = \mathbf{C} \mathbf{w}$, show that $\mathbf{0}$ (zero vector in $\mathbb{R}^d$) is a global minimum. What is the optimal value?*

#### <span style="color:rgb(46,139,87)">Answers to question 1</span> 

...

#### <span style="color:rgb(46,139,87)">Back to the problem</span> 

To define the matrix $\mathbf{C}$ above, 

1) We first build a diagonal matrix $\mathbf{D}$ with diagonal coefficients (corresponding to its eigenvalues) distributed in [$\mu$,$L$], with $0 < \mu \le L$, so that $\mu = \lambda_1 \le \cdots \le \lambda_d = L$. 

2) We then generate a random orthogonal matrix $\mathbf{Q}$ (i.e. a rotation/a change of basis) and we set
$$
    \mathbf{C}=\mathbf{Q}^T \mathbf{D} \mathbf{Q} .
$$

It follows that $\mathbf{C} = \nabla^2 f(\mathbf{w}) \succeq \mu \mathbf{I}$, and thus the resulting problem 
$\mathrm{minimize}_{\mathbf{w} \in \mathbb{R}^d} \tfrac{1}{2}\mathbf{w}^T \mathbf{C} \mathbf{w}$ is $\mu$-strongly convex.

In [ ]:
# Defining the quadratic problem
# NB: This construction is not critical for this course

# Problem dimension
d=100

# Largest eigenvalue
L=1

# Smallest eigenvalue/strong convexity constant
mu=0.01

# Fixing the random seed generator
np.random.seed(1)

# Random orthogonal matrix Q
M = np.random.multivariate_normal(np.zeros(d),np.identity(d),size=d)
Q,R = qr(M) 

# Spreading eigenvalues between mu and L
D = np.random.uniform(mu,L,d)
D = 10.**D
D = (D-min(D))/(max(D)-min(D))
D = mu+(L-mu)*D


# Final matrix/Hessian
C = Q.T @ np.diag(D) @ Q

## <span style="color:rgb(46,139,87)">1.2 - First version of gradient descent</span>

We now introduce a first version of gradient descent for our quadratic problem of interest. Since the solution of the quadratic problem is the only point with zero gradient, any point with nonzero gradient is not a minimum. Furthermore, one can show that the function $f$ can decrease in the direction of $-\nabla f(\mathbf{w})$ when this vector is nonzero.

These considerations give rise to an iterative scheme called *gradient descent*. The method starts at a point $\mathbf{w}_0$ and performs the recursion
$$
    \mathbf{w}_{k+1} = \mathbf{w}_k - \alpha_k \nabla f(\mathbf{w}_k),
$$
where $\alpha_k>0$ is called the *stepsize* at iteration $k$.

### <span style="color:rgb(46,139,87)">Question 2</span> 

*The code below implements gradient descent on the quadratic problem using a constant stepsize $\alpha_k=\alpha>0$. Fill out the missing parts in the code below.*

In [ ]:
def gd_quad(w0,C,alpha,n_iter=100, verbose=False): 
    """
        Gradient descent on quadratic functions.
        
        Inputs:
            w0: Initial point
            C: Matrix defining the quadratic problem
            alpha: Constant value for the stepsize
            n_iter: Number of iterations
            verbose: Plotting information about the run?
            
        Outputs:
            w_output: Final iterate
            objvals: History of function values (n_iter+1 values)
            
    """
    
    ############
    # Initialization
    ############

    # History of function values
    objvals = []
    
    # Initializing the iterate  
    w = w0.copy()

    # Initialization of iteration index
    k=0    
    
    ##########################################
    # FILL OUT THE MISSING LINES BELOW
    ##########################################
    
    # Initial objective value
    obj = 
    
    # Initial gradient value
    g = 

    ##########################################
    # END LINES TO BE FILLED OUT
    ##########################################
    
    # Store initial function value
    objvals.append(obj)
    
    # Optional display
    if verbose:
        print("Gradient descent:")
        print(' | '.join([name.center(8) for name in ["iter", "fval"]]))
        print(' | '.join([("%d" % k).rjust(8),("%.2e" % obj).rjust(8)]))
    
    ####################
    # Main loop
    ####################
    while (k < n_iter):
        
        ##################################
        # FILL OUT THE MISSING CODE BELOW
        ##################################
        
        
        # Update the iterate
        w[:] = 
        
        # Compute new gradient
        g = 
        
        # Compute new objective value
        obj = 
        
        
        ##################################
        # END CODE TO BE FILLED OUT
        ##################################
        
        # Store new function value
        objvals.append(obj)
        
        # Optional display
        if verbose:
            print(' | '.join([("%d" % k).rjust(8),("%.2e" % obj).rjust(8)]))       
        
        # Increment iteration index
        k += 1
    
    ######################
    # End main loop
    ######################
    
    # Output
    w_output = w.copy()
    return w_output, np.array(objvals)

### <span style="color:rgb(46,139,87)">Question 3</span> 


*a) Run the block below to test gradient descent for 100 iterations using a fixed initial point but 5 different stepsizes. Is it possible to find a value $\alpha>1$ for which the method converges? Can we find a stepsize $\alpha<1$ for which the method does not converge? What do we observe for the value $\alpha=1$?*

*b) Redo the comparison using the same 5 stepsizes as in question a), but with a budget of $1000$ iterations. Does this change the conclusions?*

*c) It can be shown that the value $\frac{2}{\mu+L}$ is the best choice for this problem in theory. Is this confirmed by the numerics?*

In [ ]:
# Answer question 3.a)

w_0 = np.ones(d)
n_iter = 100
##################
# 
vals_stepsize = [] # Test your own values here
nvals = len(vals_stepsize)

objs = np.zeros((n_iter+1,nvals))

for i_val in range(nvals):
    _, objs[:,i_val] = gd_quad(w_0,C,vals_stepsize[i_val],n_iter, verbose=False)


In [ ]:
# Plotting the comparison from the block above
# x-axis : Number of iterations
# y-axis : Function value
plt.figure(figsize=(7,5))
plt.set_cmap("RdPu")
for i_val in range(nvals):
    plt.semilogy(objs[:,i_val], label="GD("+str(vals_stepsize[i_val])+")", lw=2)
plt.title("Convergence", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective value (log)", fontsize=14)
plt.legend(loc=3)

In [ ]:
# Answer to question 3.b

w_0 = np.ones(d)
n_iter = 1000
##################
# 
vals_stepsize = [] # Test your own values here (same than 3.a)
nvals = len(vals_stepsize)

objs = np.zeros((n_iter+1,nvals))

for i_val in range(nvals):
    _, objs[:,i_val] = gd_quad(w_0,C,vals_stepsize[i_val],n_iter, verbose=False)

In [ ]:
# Plotting the comparison from the block above
# x-axis : Number of iterations
# y-axis : Function value
plt.figure(figsize=(7,5))
plt.set_cmap("RdPu")
for i_val in range(nvals):
    plt.semilogy(objs[:,i_val], label="GD("+str(vals_stepsize[i_val])+")", lw=2)
plt.title("Convergence", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective value (log)", fontsize=14)
plt.legend(loc=3)

In [ ]:
# Answer to question 3.c using several values

w_0 = np.ones(d)
n_iter = 1000
##################
# 
vals_stepsize = [2/(L+mu),] # Use other values for comparison (e.g. 1)
nvals = len(vals_stepsize)

objs = np.zeros((n_iter+1,nvals))

for i_val in range(nvals):
    _, objs[:,i_val] = gd_quad(w_0,C,vals_stepsize[i_val],n_iter, verbose=False)


In [ ]:
# Plot the comparison for question 3-c
# x-axis : Number of iterations
# y-axis : Function value
plt.figure(figsize=(7,5))
plt.set_cmap("RdPu")
for i_val in range(nvals):
    if i_val==0:
        plt.semilogy(objs[:100,i_val], label="GD(2/(L+mu))", lw=2)
    else:
        plt.semilogy(objs[:100,i_val], label="GD("+str(vals_stepsize[i_val])+")", lw=2)
plt.title("Convergence (100 its)", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective function (log)", fontsize=14)
plt.legend(loc=3)

plt.figure(figsize=(7,5))
plt.set_cmap("RdPu")
for i_val in range(nvals):
    if i_val==0:
        plt.semilogy(objs[:,i_val], label="GD(2/(L+mu))", lw=2)
    else:
        plt.semilogy(objs[:,i_val], label="GD("+str(vals_stepsize[i_val])+")", lw=2)
plt.title("Convergence ("+str(n_iter)+" its)", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective function (log)", fontsize=14)
plt.legend(loc=3)

# <span style="color:rgb(46,139,87)">Part 2 - Convex regression problems</span>

We now consider two classes of regression problems, that are typical examples of data science problems. We will adress those using a generic gradient descent implementation.

## <span style="color:rgb(46,139,87)">2.1 - Dataset and tasks</span>

We consider a dataset of the form $\{(\mathbf{x}_i,y_i)\}_{i=1,\dots,n}$, where $\mathbf{x}_i \in \mathbb{R}^d$ and $y_i \in \mathbb{R}$. We store this dataset using

- an input matrix $\mathbf{X} \in \mathbb{R}^{n \times d}$;
- and an output vector $\mathbf{y} \in \mathbb{R}^n$. 

The dataset will be generated using the routine below, that controls an amount of noise introduced in the data.

**This code is not critical for the course. Students are not expected to produce similar pieces of code.**

In [ ]:
# Data generation (code inspired by A. Gramfort from INRIA).

def simu_linmodel(w, n, std=1., corr=0.5):
    
    """
    Linear model with additive noise
    
    Inputs
    ----------
        w: Model coefficients
    
        n: Number of samples
    
        std: Standard deviation
        
        corr: Correlation coefficient
        
    Outputs
    ----------
    
        X: Input matrix
        y: Output vector
    """    
    
    d = w.shape[0]
    cov = toeplitz(corr ** np.arange(0, d))
    X = multivariate_normal(np.zeros(d), cov, size=n)
    noise = std * randn(n)
    y = X.dot(w) + noise
    return X, y

The data is obtained by a linear model corrupted with gaussian noise.

Our goal is to recover a linear trend from the data, i.e. we seek a linear function $\mathbf{x} \mapsto \mathbf{x}^T \mathbf{w}$ parameterized by $\mathbf{w}$ that fits the data in the sense of a loss function $\ell$. This gives rise to the following optimization problem
$$
    \mathrm{minimize}_{\mathbf{w} \in \mathbb{R}^d} f(\mathbf{w}) 
    = \frac{1}{n} \sum_{i=1}^n f_i(\mathbf{w}), 
    \qquad f_i(\mathbf{w}) = \ell(\mathbf{x}_i^T\mathbf{w},y_i) 
    + \frac{\lambda}{2}\|\mathbf{w}\|^2.
$$
where $\lambda \ge 0$ is a regularization parameter *(more on this later)*.

We will focus on two specific instances of this problem, for which $f$ is $\mathcal{C}^1$, convex, and has an $L$-Lipschitz continuous gradient, that is
$$
    \|\nabla f(\mathbf{w})-\nabla f(\mathbf{v})\| \le L \|\mathbf{w}-\mathbf{v}\|.
$$
As we will see during the next session, this property allows to determine good stepsize rules for gradient descent.

### <span style="color:rgb(46,139,87)">Linear regression</span>

In linear regression, we consider the $\ell_2$ loss and the problem
$$
    \mathrm{minimize}_{\mathbf{w} \in \mathbb{R}^d} f(\mathbf{w}) 
    := \frac{1}{2 n} \|\mathbf{X} \mathbf{w} - \mathbf{y}\|^2 + \frac{\lambda}{2}\|\mathbf{w}\|^2.
$$ 

- The function $f$ is quadratic hence $\mathcal{C}^1$. Its gradient at $\mathbf{w} \in \mathbb{R}^d$ is
$$
    \nabla f(\mathbf{w}) = \frac{1}{n}\mathbf{X}^T (\mathbf{X} \mathbf{w} - \mathbf{y}) + \lambda \mathbf{w}.
$$
 
- The function $f$ is convex, and even $(\sigma_{\min}(\mathbf{X})^2+\lambda)$-strongly convex when $\sigma_{\min}(\mathbf{X})^2+\lambda>0$, where $\sigma_{\min}(\mathbf{X})$ is the smallest singular value of $\mathbf{X}$.

- The gradient of $f$ is $L$-Lipschitz continuous with $L = \frac{\|\mathbf{X}^T \mathbf{X}\|}{n}+\lambda$. 

### <span style="color:rgb(46,139,87)">Logistic regression</span>

In logistic regression, we still consider a linear model, but for binary classification purposes, assuming $y_i \in \{-1,1\}$. The corresponding problem is given by
$$
    \mathrm{minimize}_{\mathbf{w} \in \mathbb{R}^d} f(\mathbf{w}) 
    := \frac{1}{n} \sum_{i=1}^n f_i(\mathbf{w}), \qquad 
    f_i(\mathbf{w})=\log(1+\exp(-y_i \mathbf{x}_i^T \mathbf{w}))+\frac{\lambda}{2}\|\mathbf{w}\|^2.
$$

- The function $f$ is $\mathcal{C}^{1}$. For every $\mathbf{w} \in \mathbb{R}^d$, we have
$$
\nabla f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n  -\frac{y_i}{1 + \exp(y_i \mathbf{x}_i^T \mathbf{w})} \mathbf{x}_i + \lambda \mathbf{w}.
$$

- The function $f$ is convex, and even $\lambda$-strongly convex when $\lambda>0$.

- The gradient of $f$ is $L$-Lipschitz continuous with $L =\frac{\|\mathbf{A}^T \mathbf{A}\|}{4n}+\lambda$.

## <span style="color:rgb(46,139,87)">2.2 Python class and instances</span>

We now implement the two problems above in a generic Python class, that will allow to query objective values, gradients, as well as Lipschitz constants for the gradient and strong convexity constants.

In [ ]:
# Python class for regression problems
class RegPb(object):
    '''
        Python class for regression problems with linear models
        
        Attributes:
            X: Data input matrix
            y: Data output vector
            n,d: Dimensions of X
            loss: Chosen loss function
                'l2': Least-squares loss (typically for linear regression)
                'logit': Logistic loss (typically for logistic regression)
            lbda: Regularization parameter
    '''
   
    # Initialization
    def __init__(self, X, y,lbda=0,loss='l2'):
        self.X = X
        self.y = y
        self.n, self.d = X.shape
        self.loss = loss
        self.lbda = lbda
    
    # Objective function
    def fun(self, w):
        if self.loss=='l2':
            return norm(self.X.dot(w) - self.y) ** 2 / (2. * self.n) + self.lbda * norm(w) ** 2 / 2.
        elif self.loss=='logit':
            yXw = self.y * self.X.dot(w)
            return np.mean(np.log(1. + np.exp(-yXw))) + self.lbda * norm(w) ** 2 / 2.

    
    # Gradient vector
    def grad(self, w):
        if self.loss=='l2':
            return self.X.T.dot(self.X.dot(w) - self.y) / self.n + self.lbda * w
        elif self.loss=='logit':
            yXw = self.y * self.X.dot(w)
            aux = 1. / (1. + np.exp(yXw))
            return - (self.X.T).dot(self.y * aux) / self.n + self.lbda * w
    
    # Lipschitz constant for the gradient
    def lipgrad(self):
        if self.loss=='l2':
            L = norm(self.X, ord=2) ** 2 / self.n + self.lbda
        elif self.loss=='logit':
            L = norm(self.X, ord=2) ** 2 / (4. * self.n) + self.lbda
        return L
    
    # ''Strong'' convexity constant (can be 0 when self.lbda=0)
    def cvxval(self):
        if self.loss=='l2':
            s = svdvals(self.X)
            mu = min(s)**2 / self.n 
            return mu + self.lbda
        elif self.loss=='logit':
            return self.lbda

We now generate one instance of linear regression and one instance of logistic regression based on our dataset generation technique from Part 2.1 of the notebook. 

In [ ]:
# Generating two relatively small instances

d = 50
n = 1000
idx = np.arange(d)
lbda = 1. / n ** (0.5)

# Fixing the random number generator
np.random.seed(1)

# Coefficients of the true solution
w_model_truth = (-1)**idx * np.exp(-idx / 10.)

Xlin, ylin = simu_linmodel(w_model_truth, n, std=1., corr=0.1)
Xlog, ylog = simu_linmodel(w_model_truth, n, std=1., corr=0.7)
ylog = np.sign(ylog) # Taking signs for binary classification

pblinreg = RegPb(Xlin, ylin,lbda,loss='l2')
pblogreg = RegPb(Xlog, ylog,lbda,loss='logit')

### <span style="color:rgb(46,139,87)">Estimating `min` and `argmin`</span>

Unlike in the first part of this notebook, we cannot compute the solutions to both regression problems by hand. However, since the problem size is relatively small, one can use efficient numerical techniques *(more on those in subsequent lectures)* to compute good approximations for the optimal value $f^*$ and an optimal solution $\mathbf{w}^*$. Using those values, we will then be able to plot the distances to optimality in terms of function value, namely $f(\mathbf{w}_k)-f^*$, and in terms of parameters, namely $\|\mathbf{w}_k -\mathbf{w}^*\|$.

In [ ]:
# Computing a good approximation of the optimal value and solution
# Uses the optimization method L-BFGS-B

w_init = np.zeros(d)

# Linear regression
w_min_lin, f_min_lin, _ = fmin_l_bfgs_b(pblinreg.fun, w_init, pblinreg.grad, args=(), pgtol=1e-30, factr =1e-30)
print("Linear regression:")
print("\t Numerical optimal value:",f_min_lin)
print("\t Gradient norm at numerical optimum:",norm(pblinreg.grad(w_min_lin)))

# Logistic regression
w_min_log, f_min_log, _ = fmin_l_bfgs_b(pblogreg.fun, w_init, pblogreg.grad, args=(), pgtol=1e-30, factr =1e-30)
print("Logistic regression:")
print("\t Numerical optimal value:",f_min_log)
print("\t Gradient norm at numerical optimum:",norm(pblogreg.grad(w_min_log)))

## <span style="color:rgb(46,139,87)">2.3 Gradient descent</span>

The block below presents a generic gradient descent scheme, that is relatively similar to the specific one of Part 1. However, it is more generic, and applicable to any problem class possessing the same attributes than our regression problem class.

On the algorithmic side, the code aims at providing three strategies for choosing the stepsize, namely:

- *Constant stepsize:* $\alpha_k = \frac{\bar{\alpha}}{L}$, where $L$ is the Lipschitz constant of $\nabla f$ (computed within the problem class) and $\bar{\alpha}>0$ is an algorithmic hyperparameter.

- *Decreasing stepsize:* $\alpha_k = \frac{\bar{\alpha}}{(k+1)^a}$, where $a>0$ and $\bar{\alpha}>0$ are algorithmic hyperparameters.

- *Line search:* $\alpha_k=\frac{\bar{\alpha}}{2^{j_k}}$, where $j_k$ is the smallest nonnegative integer such that
$$
f\left(\mathbf{w}_k-\frac{\bar{\alpha}}{2^{j_k}}\nabla f(\mathbf{w}_k)\right) < f(\mathbf{w}_k) - 0.0001 \frac{\alpha}{2^{j_k}}\|\nabla f(\mathbf{w}_k)\|^2.
$$
In practice, line search stops if the condition is satisfied or if $\frac{\bar{\alpha}}{2^{j_k}} < 10^{-10}$ (that is, the stepsize drops below a certain threshold). The values $2$,$0.0001$ and $10^{-10}$ are fixed for simplicity, but could be hyperparameters of the algorithm, like $\bar{\alpha}>0$.


### <span style="color:rgb(46,139,87)">Question 4</span> 

*Fill out the code below with the three stepsize choices described above (read the algorithm description).*

In [ ]:
# Gradient descent
def gd_reg(w0,problem,wopt,alphachoice=0,alphabar=1, n_iter=1000, verbose=False): 

    """
        Implementing gradient descent with various
        stepsize choices.

        Inputs:

            w0: Initial point
            problem: Problem instance
                problem.fun(w) Objective function at w
                problem.grad(w) Gradient vector at w
                problem.lipgrad() Lipschitz constant for the gradient
            wopt: Target minimum value for the method
            alphachoice: Stepsize strategy
                0: Constant, proportional 1/L
                a>0: Decreasing in 1/((k+1)**a)
                -1: Line search
            alphabar: Initial stepsize
            n_iter: Maximum number of iterations
            verbose: Iteration display (optional)

        Outputs:

            w_output: Last iterate
            objvals: History of function values (length n_iter+1)
            distits: History of distances to optimum (length n_iter+1)
            ngvals: History of gradient norms (length n_iter)
            
    """

    ############
    # Initialization

    # History
    objvals = []
    ngvals = []
    distits = []
    
    # Lipschitz constant for the gradient
    L = problem.lipgrad()
    
    # Initial vector
    w = w0.copy()

    # Iteration index
    k=0    
    
    # Initial function value
    obj = problem.fun(w) 
    objvals.append(obj);
    
    # Initial gradient
    g = problem.grad(w)
    ng = norm(g)
    ngvals.append(ng)
    
    # Distance to target point
    dist = norm(w-wopt)
    distits.append(dist)

    # Optional display
    if verbose:
        print("Gradient descent:")
        print(' | '.join([name.center(8) for name in ["iter", "fval", "dist","stepsize"]]))
    
    ####################
    # Main loop
    while (k < n_iter):
        
        #####################################################################################
        # FILL OUT THIS PART
        
        # 1 - Define stepsize s as a function of k, L, alpha0 and g
        if alphachoice==0:
            # Constant stepsize
            s = 
        elif alphachoice>0:
            # Decreasing stepsize
            s = 
        elif alphachoice==-1:
            # Line search
            s = 
        
        # 2 - Gradient descent iteration with stepsize s
        w[:] = 
      
            
        # END PART TO BE FILLED OUT
        # w should contain the new iterate and s should contain the
        # current stepsize value
        ####################################################################################
        
        
        # Optional display
        if verbose:        
            print(' | '.join([("%d" % k).rjust(8),("%.2e" % obj).rjust(8),("%.2e" % dist).rjust(8),("%.2e" % s).rjust(8)]))
        
        # Compute useful quantities for the next iteration
        obj = problem.fun(w)
        objvals.append(obj)
        g = problem.grad(w)
        ng = norm(g)
        ngvals.append(ng)
        dist = norm(w-wopt)
        distits.append(dist)
        
        # Increment iteration index
        k += 1
    
    # End main loop
    ######################    

    # Output

    w_output = w.copy()

    return w_output, np.array(objvals), np.array(distits), np.array(ngvals)

### <span style="color:rgb(46,139,87)">Question 5</span> 

*We consider first the linear regression instance.*

*a) Run the block below to test gradient descent with the following stepsize choices:*

- $\alpha_k = \frac{1}{L}$;

- $\alpha_k = \frac{1}{k+1}$;

- $\alpha_k = \frac{1}{\sqrt{k+1}}$;

- $\alpha_k$ *line search with* $\bar{\alpha}=1$.

*b) Compare the curves of $\{f(x_k)\}$ and $\{f(x_k)-f^*_{lin}\}$, where
$f^*_{lin}$ is the optimal value approximation computed numerically.*

*c) Plot the gradient norms as well as the distances to optimality as a function of the iteration index. Are those plots consistent with that of question a)?* 

*d) What quantity does not appear in the previous plots, thus preventing a fair comparison between them? Hint: Think about what line search is doing.*

In [ ]:
# Answer to question 5.a)

# Run four variants of gradient descent
w0 = np.zeros(d)
_, obj_a, dist_a, ngrad_a = gd_reg(w0,pblinreg,w_min_lin,alphachoice=0,alphabar=1, n_iter=50)
_, obj_b, dist_b, ngrad_b = gd_reg(w0,pblinreg,w_min_lin,alphachoice=1,alphabar=1, n_iter=50)
_, obj_c, dist_c, ngrad_c = gd_reg(w0,pblinreg,w_min_lin,alphachoice=0.5,alphabar=1, n_iter=50)
_, obj_d, dist_d, ngrad_d = gd_reg(w0,pblinreg,w_min_lin,alphachoice=-1,alphabar=1, n_iter=50)

# Final objective value
print("Final objective value - 1/L",obj_a[-1])
print("Final objective value - 1/(k+1)",obj_b[-1])
print("Final objective value - 1/(sqrt(k+1))",obj_c[-1])
print("Final objective value - Line search",obj_d[-1])

In [ ]:
# Answer to question 5.b)

# Objective value
plt.figure(figsize=(7, 5))
plt.semilogy(obj_a, label="GD - 1/L", lw=2)
plt.semilogy(obj_b, label="GD - 1/(k+1)", lw=2)
plt.semilogy(obj_c, label="GD - 1/(sqrt(k+1))", lw=2)
plt.semilogy(obj_d, label="GD - Line search", lw=2)
plt.title("Change in objective value", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective function (log)", fontsize=14)
plt.legend()

# We can also plot the change in function value relatively to the target value 
# Note: In log scale, values close to $0$ do not appear or produce 
# artifacts.


#### Relative objective function
plt.figure(figsize=(7, 5))
plt.semilogy(obj_a-f_min_lin, label="GD - 1/L", lw=2)
plt.semilogy(obj_b-f_min_lin, label="GD - 1/(k+1)", lw=2)
plt.semilogy(obj_c-f_min_lin, label="GD - 1/(sqrt(k+1))", lw=2)
plt.semilogy(obj_d-f_min_lin, label="GD - Line search", lw=2)
plt.title("Relative change in objective value", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective-Target (log)", fontsize=14)
plt.legend()

In [ ]:
# Answer to question 5.c)

#### Gradient norms
plt.figure(figsize=(7, 5))
plt.semilogy(ngrad_a, label="GD - 1/L", lw=2)
plt.semilogy(ngrad_b , label="GD - 1/(k+1)", lw=2)
plt.semilogy(ngrad_c, label="GD - 1/(sqrt(k+1))", lw=2)
plt.semilogy(ngrad_d, label="GD - Line search", lw=2)
plt.title("Convergence of gradient norm", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Gradient norm (log)", fontsize=14)
plt.legend()

#### Plot the behavior of all four methods in terms of distance to numerical optimum
plt.figure(figsize=(7, 5))
plt.semilogy(dist_a, label="GD - 1/L", lw=2)
plt.semilogy(dist_b , label="GD - 1/(k+1)", lw=2)
plt.semilogy(dist_c, label="GD - 1/(sqrt(k+1))", lw=2)
plt.semilogy(dist_d, label="GD - Line Search", lw=2)
plt.title("Distance to numerical optimum", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Distance (log)", fontsize=14)
plt.legend()

#### <span style="color:rgb(46,139,87)">Comments on question 5</span>

...

### <span style="color:rgb(46,139,87)">Question 6</span>

*The block below runs certain tests from question 5 using the logistic regression problem. Check that similar conclusions hold, and add your own instance to see whether you can beat the "Best?" one below.*

In [ ]:
# Answer to question 6)

# Testing five stepsize choices including a 'best?' one
w0 = np.zeros(d)
_, obj_a, dist_a, ngrad_a = gd_reg(w0,pblogreg,w_min_lin,alphachoice=0,alphabar=1, n_iter=50)
_, obj_b, dist_b, ngrad_b = gd_reg(w0,pblogreg,w_min_lin,alphachoice=1,alphabar=1, n_iter=50)
_, obj_c, dist_c, ngrad_c = gd_reg(w0,pblogreg,w_min_lin,alphachoice=0.5,alphabar=1, n_iter=50)
_, obj_d, dist_d, ngrad_d = gd_reg(w0,pblogreg,w_min_lin,alphachoice=-1,alphabar=1, n_iter=50)
_, obj_e, dist_e, ngrad_e = gd_reg(w0,pblogreg,w_min_lin,alphachoice=-1,alphabar=10, n_iter=50)


#### Function value curves
plt.figure(figsize=(7, 5))
plt.semilogy(obj_a-f_min_lin, label="GD - 1/L", lw=2)
plt.semilogy(obj_b-f_min_lin, label="GD - 1/(k+1)", lw=2)
plt.semilogy(obj_c-f_min_lin, label="GD - 1/(sqrt(k+1))", lw=2)
plt.semilogy(obj_d-f_min_lin, label="GD - Line Search", lw=2)
plt.semilogy(obj_e-f_min_lin, label="GD - Best?", lw=2)

plt.title("Relative objective values", fontsize=16)
plt.xlabel("#Iterations", fontsize=14)
plt.ylabel("Objective-Target (log)", fontsize=14)
plt.legend()

### <span style="color:rgb(46,139,87)">Question 7 (optional)</span>
To provide a fair comparision between stepsize strategies, (_c.f._ question d)), adapt the implementation of function `gd_reg`.

...